In [1]:
from collections import deque      
from heapq import heappush, heappop  
from copy import deepcopy   
from typing import List, Tuple, Optional


In [2]:
#Goal States and Start States
goal_state = [
    [1, 2, 3],
    [7, 8, 0],
    [4, 5, 6],
]

start_state1 = [
    [5, 4, 0],
    [6, 1, 8],
    [7, 3, 2],
]

start_state2 = [
    [4, 5, 8],
    [3, 1, 0],
    [7, 6, 2],
]




In [3]:
#helper functions
def find_blank(state:List[List[int]]) -> Tuple[int, int]:
    '''Find the pos of blank 0 state'''
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return (i, j)
            
#further mentions of list is already defined
def is_goal(state) -> bool:
    return state == goal_state

#generate new neighbors from current state
def get_neighbors(state) -> List[List[List[int]]]:
    neighbors = []
    x, y = find_blank(state)
    moves = [(-1,0),(1,0),(0,-1),(0,1)]  # up, down, left, right
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [row[:] for row in state]  # copy
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            neighbors.append(new_state)
    return neighbors

#
def state_to_tuple(state) -> Tuple[Tuple[int, int, int], Tuple[int, int, int], Tuple[int, int, int]]:
    '''Convert list state to tuple for hashing'''
    return tuple(tuple(row) for row in state)


In [4]:
#print utils

def display_solution(solution: Optional[List[List[List[int]]]], total_cost: int = -1) -> None:
    """
    Display the solution path for an 8-puzzle solver with numbered steps.
    """
    if solution:
        move_count = len(solution) - 1
        if total_cost >= 0:
            print(f"Solution found in {move_count} moves with total cost {total_cost}:")
        else:
            print(f"Solution found in {move_count} moves:")

        for step, state in enumerate(solution, start=0):  # step counter
            print(f"\nStep {step}:")  # numbered step
            for row in state:
                print(row)
    else:
        print("No solution found.")


In [14]:
#heuristic function
def manhattan_distance(state) -> int:
    '''calc the manhattan distance of a state to its neighbor'''
    distance = 0
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val != 0:
                goal_x, goal_y = divmod(val-1, 3)
                distance += abs(i - goal_x) + abs(j - goal_y)

    return distance

In [5]:
#BFS
def bfs(start: List[List[int]]) -> List[List[List[int]]]:
    '''BFS uses queue'''
    visited = set()
    queue = deque()
    #queue will hold pairs, start and path to that point
    queue.append((start, []))
    visited.add(state_to_tuple(start))

    while queue:
        state, path = queue.popleft()
        if is_goal(state):
            return path + [state]
        
        for neighbor in get_neighbors(state):
            t_neighbor = state_to_tuple(neighbor)
            if t_neighbor not in visited:
                visited.add(t_neighbor)
                queue.append((neighbor, path + [state]))
    
    return None #no solution




In [6]:
def dfs(start: List[List[int]], max_depth: int) -> List[List[List[int]]]:
    '''DFS uses a stack'''
    visited = set()
    stack = [(start, [], 0)]  # state, path, depth

    while stack:
        state, path, depth = stack.pop()

        if is_goal(state):
            return path + [state]

        if depth >= max_depth:
            continue

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            # reverse to simulate left-to-right exploration
            for neighbor in reversed(get_neighbors(state)):
                stack.append((neighbor, path + [state], depth + 1))

    return None  # if no solution found

In [7]:
#UCS
def ucs(start):
    visited = set()
    pq = []
    heappush(pq, (0, start, []))  # cost, state, path

    while pq:
        cost, state, path = heappop(pq)

        if is_goal(state):
            return path + [state], cost

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            for neighbor in get_neighbors(state):
                heappush(pq, (cost + 1, neighbor, path + [state]))
        
    return None, None


In [8]:
#IDS

def dls_ids(state, limit, visited = set()):
    stack =  [(state, [], 0)] # state, path, depth
    while stack:
        current_state, path, depth = stack.pop()
        if is_goal(current_state):
            return path + [current_state]
        if depth >= limit:
            continue
        t_state = state_to_tuple(current_state)
        if t_state not in visited:
            visited.add(t_state)

            for neighbor in reversed(get_neighbors(current_state)):
                stack.append((neighbor, path + [current_state], depth + 1))
    return None

def ids(start, max_depth = 50):
    for depth in range(max_depth):
        visited = set()
        result = dls_ids(start, depth, visited)
        if result is not None:
            return result
    return None

In [17]:
def gbfs(start):
    visited = set()
    pq = []
    heappush(pq, (manhattan_distance(start), start, []))  # heuristic, state, path

    counter = 1
    while pq:
        h , state, path = heappop(pq)

        print("Expanding node {} with heuristic {}".format(counter, h))

        if is_goal(state):
            return path + [state]
        
        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            for neighbor in get_neighbors(state):
                h_val = manhattan_distance(neighbor)
                heappush(pq, (h_val, neighbor, path + [state]))

        counter += 1

    return None

In [24]:
def astar(start):
    visited = set()
    pq = []
    g_cost = 0
    h_cost = manhattan_distance(start)
    f_cost = g_cost + h_cost
    heappush(pq, (f_cost, g_cost, start, []))  # f_cost, g_cost, state, path

    counter = 0

    while pq:
        f, g, state, path = heappop(pq)

        h = f - g
        
        print("Expanding node {} with g: {}, h: {}, f: {}".format(counter, g, h, f))

        if is_goal(state):
            return path + [state], g
        
        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            for neighbor in get_neighbors(state):
                g_new = g + 1
                h_new = manhattan_distance(neighbor)
                f_new = g_new + h_new
                heappush(pq, (f_new, g_new, neighbor, path + [state]))

        counter += 1
        
    return None, None


--FROM THIS POINT ON ITS ALL THE RUNS--

In [9]:
#bfs runs
print("\n---------------------")
print("BFS on Start Goal 1")
print("---------------------")
bfs_sol1 = bfs(start_state1)
display_solution(bfs_sol1)

print("\n---------------------")
print("BFS on Start Goal 2")
print("---------------------")
bfs_sol2 = bfs(start_state2)
display_solution(bfs_sol2)


---------------------
BFS on Start Goal 1
---------------------
Solution found in 23 moves:

Step 0:
[5, 4, 0]
[6, 1, 8]
[7, 3, 2]

Step 1:
[5, 4, 8]
[6, 1, 0]
[7, 3, 2]

Step 2:
[5, 4, 8]
[6, 1, 2]
[7, 3, 0]

Step 3:
[5, 4, 8]
[6, 1, 2]
[7, 0, 3]

Step 4:
[5, 4, 8]
[6, 1, 2]
[0, 7, 3]

Step 5:
[5, 4, 8]
[0, 1, 2]
[6, 7, 3]

Step 6:
[0, 4, 8]
[5, 1, 2]
[6, 7, 3]

Step 7:
[4, 0, 8]
[5, 1, 2]
[6, 7, 3]

Step 8:
[4, 1, 8]
[5, 0, 2]
[6, 7, 3]

Step 9:
[4, 1, 8]
[5, 7, 2]
[6, 0, 3]

Step 10:
[4, 1, 8]
[5, 7, 2]
[0, 6, 3]

Step 11:
[4, 1, 8]
[0, 7, 2]
[5, 6, 3]

Step 12:
[0, 1, 8]
[4, 7, 2]
[5, 6, 3]

Step 13:
[1, 0, 8]
[4, 7, 2]
[5, 6, 3]

Step 14:
[1, 8, 0]
[4, 7, 2]
[5, 6, 3]

Step 15:
[1, 8, 2]
[4, 7, 0]
[5, 6, 3]

Step 16:
[1, 8, 2]
[4, 7, 3]
[5, 6, 0]

Step 17:
[1, 8, 2]
[4, 7, 3]
[5, 0, 6]

Step 18:
[1, 8, 2]
[4, 7, 3]
[0, 5, 6]

Step 19:
[1, 8, 2]
[0, 7, 3]
[4, 5, 6]

Step 20:
[1, 8, 2]
[7, 0, 3]
[4, 5, 6]

Step 21:
[1, 0, 2]
[7, 8, 3]
[4, 5, 6]

Step 22:
[1, 2, 0]
[7, 8, 3]
[4, 5, 

In [10]:
#dfs runs
print("\n---------------------")
print("DFS on Start Goal 1")
print("---------------------")
dfs_sol1 = dfs(start_state1,1500)
display_solution(dfs_sol1)

print("\n---------------------")
print("DFS on Start Goal 2")
print("---------------------")
dfs_sol2 = dfs(start_state2,300)
display_solution(dfs_sol2)


---------------------
DFS on Start Goal 1
---------------------
Solution found in 1485 moves:

Step 0:
[5, 4, 0]
[6, 1, 8]
[7, 3, 2]

Step 1:
[5, 4, 8]
[6, 1, 0]
[7, 3, 2]

Step 2:
[5, 4, 8]
[6, 1, 2]
[7, 3, 0]

Step 3:
[5, 4, 8]
[6, 1, 2]
[7, 0, 3]

Step 4:
[5, 4, 8]
[6, 0, 2]
[7, 1, 3]

Step 5:
[5, 0, 8]
[6, 4, 2]
[7, 1, 3]

Step 6:
[0, 5, 8]
[6, 4, 2]
[7, 1, 3]

Step 7:
[6, 5, 8]
[0, 4, 2]
[7, 1, 3]

Step 8:
[6, 5, 8]
[7, 4, 2]
[0, 1, 3]

Step 9:
[6, 5, 8]
[7, 4, 2]
[1, 0, 3]

Step 10:
[6, 5, 8]
[7, 0, 2]
[1, 4, 3]

Step 11:
[6, 0, 8]
[7, 5, 2]
[1, 4, 3]

Step 12:
[0, 6, 8]
[7, 5, 2]
[1, 4, 3]

Step 13:
[7, 6, 8]
[0, 5, 2]
[1, 4, 3]

Step 14:
[7, 6, 8]
[1, 5, 2]
[0, 4, 3]

Step 15:
[7, 6, 8]
[1, 5, 2]
[4, 0, 3]

Step 16:
[7, 6, 8]
[1, 0, 2]
[4, 5, 3]

Step 17:
[7, 0, 8]
[1, 6, 2]
[4, 5, 3]

Step 18:
[0, 7, 8]
[1, 6, 2]
[4, 5, 3]

Step 19:
[1, 7, 8]
[0, 6, 2]
[4, 5, 3]

Step 20:
[1, 7, 8]
[4, 6, 2]
[0, 5, 3]

Step 21:
[1, 7, 8]
[4, 6, 2]
[5, 0, 3]

Step 22:
[1, 7, 8]
[4, 0, 2]
[5, 6

In [11]:
# ucs runs
print("\n---------------------")
print("UCS on Start State 1")
print("---------------------")
ucs_sol1, ucs_cost1 = ucs(start_state1)
display_solution(ucs_sol1, ucs_cost1 if ucs_sol1 else -1)

print("\n---------------------")
print("UCS on Start State 2")
print("---------------------")
ucs_sol2, ucs_cost2 = ucs(start_state2)
display_solution(ucs_sol2, ucs_cost2 if ucs_sol2 else -1)


---------------------
UCS on Start State 1
---------------------
Solution found in 23 moves with total cost 23:

Step 0:
[5, 4, 0]
[6, 1, 8]
[7, 3, 2]

Step 1:
[5, 0, 4]
[6, 1, 8]
[7, 3, 2]

Step 2:
[5, 1, 4]
[6, 0, 8]
[7, 3, 2]

Step 3:
[5, 1, 4]
[0, 6, 8]
[7, 3, 2]

Step 4:
[0, 1, 4]
[5, 6, 8]
[7, 3, 2]

Step 5:
[1, 0, 4]
[5, 6, 8]
[7, 3, 2]

Step 6:
[1, 4, 0]
[5, 6, 8]
[7, 3, 2]

Step 7:
[1, 4, 8]
[5, 6, 0]
[7, 3, 2]

Step 8:
[1, 4, 8]
[5, 6, 2]
[7, 3, 0]

Step 9:
[1, 4, 8]
[5, 6, 2]
[7, 0, 3]

Step 10:
[1, 4, 8]
[5, 0, 2]
[7, 6, 3]

Step 11:
[1, 0, 8]
[5, 4, 2]
[7, 6, 3]

Step 12:
[1, 8, 0]
[5, 4, 2]
[7, 6, 3]

Step 13:
[1, 8, 2]
[5, 4, 0]
[7, 6, 3]

Step 14:
[1, 8, 2]
[5, 4, 3]
[7, 6, 0]

Step 15:
[1, 8, 2]
[5, 4, 3]
[7, 0, 6]

Step 16:
[1, 8, 2]
[5, 0, 3]
[7, 4, 6]

Step 17:
[1, 8, 2]
[0, 5, 3]
[7, 4, 6]

Step 18:
[1, 8, 2]
[7, 5, 3]
[0, 4, 6]

Step 19:
[1, 8, 2]
[7, 5, 3]
[4, 0, 6]

Step 20:
[1, 8, 2]
[7, 0, 3]
[4, 5, 6]

Step 21:
[1, 0, 2]
[7, 8, 3]
[4, 5, 6]

Step 22:
[1, 2, 

In [12]:
#dls runs

print("\n---------------------")
print("DLS on Start Goal 1")
print("---------------------")
dls_sol1 = dfs(start_state1,30)
display_solution(dls_sol1)

print("\n---------------------")
print("DLS on Start Goal 2")
print("---------------------")
dls_sol2 = dfs(start_state2,30)
display_solution(dls_sol2)


---------------------
DLS on Start Goal 1
---------------------
No solution found.

---------------------
DLS on Start Goal 2
---------------------
No solution found.


In [13]:
print("\n---------------------")
print("IDS on Start Goal 1")
print("---------------------")
ids_sol1 = ids(start_state1)
display_solution(ids_sol1)

print("\n---------------------")
print("IDS on Start Goal 2")
print("---------------------")
ids_sol2 = ids(start_state2)
display_solution(ids_sol2)


---------------------
IDS on Start Goal 1
---------------------
Solution found in 29 moves:

Step 0:
[5, 4, 0]
[6, 1, 8]
[7, 3, 2]

Step 1:
[5, 4, 8]
[6, 1, 0]
[7, 3, 2]

Step 2:
[5, 4, 8]
[6, 0, 1]
[7, 3, 2]

Step 3:
[5, 0, 8]
[6, 4, 1]
[7, 3, 2]

Step 4:
[5, 8, 0]
[6, 4, 1]
[7, 3, 2]

Step 5:
[5, 8, 1]
[6, 4, 0]
[7, 3, 2]

Step 6:
[5, 8, 1]
[6, 4, 2]
[7, 3, 0]

Step 7:
[5, 8, 1]
[6, 4, 2]
[7, 0, 3]

Step 8:
[5, 8, 1]
[6, 0, 2]
[7, 4, 3]

Step 9:
[5, 8, 1]
[0, 6, 2]
[7, 4, 3]

Step 10:
[5, 8, 1]
[7, 6, 2]
[0, 4, 3]

Step 11:
[5, 8, 1]
[7, 6, 2]
[4, 0, 3]

Step 12:
[5, 8, 1]
[7, 0, 2]
[4, 6, 3]

Step 13:
[5, 0, 1]
[7, 8, 2]
[4, 6, 3]

Step 14:
[5, 1, 0]
[7, 8, 2]
[4, 6, 3]

Step 15:
[5, 1, 2]
[7, 8, 0]
[4, 6, 3]

Step 16:
[5, 1, 2]
[7, 8, 3]
[4, 6, 0]

Step 17:
[5, 1, 2]
[7, 8, 3]
[4, 0, 6]

Step 18:
[5, 1, 2]
[7, 8, 3]
[0, 4, 6]

Step 19:
[5, 1, 2]
[0, 8, 3]
[7, 4, 6]

Step 20:
[0, 1, 2]
[5, 8, 3]
[7, 4, 6]

Step 21:
[1, 0, 2]
[5, 8, 3]
[7, 4, 6]

Step 22:
[1, 2, 0]
[5, 8, 3]
[7, 4, 

In [18]:
print("\n---------------------")
print("GBFS on Start Goal 1")
print("---------------------")
gbfs_sol1 = gbfs(start_state1)
display_solution(gbfs_sol1)

print("\n---------------------")
print("GBFS on Start Goal 2")
print("---------------------")
gbfs_sol2 = gbfs(start_state2)
display_solution(gbfs_sol2)


---------------------
GBFS on Start Goal 1
---------------------
Expanding node 1 with heuristic 16
Expanding node 2 with heuristic 17
Expanding node 3 with heuristic 16
Expanding node 4 with heuristic 16
Expanding node 5 with heuristic 15
Expanding node 6 with heuristic 14
Expanding node 7 with heuristic 13
Expanding node 8 with heuristic 12
Expanding node 9 with heuristic 13
Expanding node 10 with heuristic 13
Expanding node 11 with heuristic 12
Expanding node 12 with heuristic 12
Expanding node 13 with heuristic 11
Expanding node 14 with heuristic 10
Expanding node 15 with heuristic 11
Expanding node 16 with heuristic 11
Expanding node 17 with heuristic 11
Expanding node 18 with heuristic 10
Expanding node 19 with heuristic 11
Expanding node 20 with heuristic 10
Expanding node 21 with heuristic 10
Expanding node 22 with heuristic 9
Expanding node 23 with heuristic 8
Expanding node 24 with heuristic 7
Expanding node 25 with heuristic 6
Expanding node 26 with heuristic 7
Expanding no

In [25]:
print("\n---------------------")
print("A* on Start Goal 1")
print("---------------------")
astar_sol1 = astar(start_state1)
display_solution(astar_sol1)

print("\n---------------------")
print("A* on Start Goal 2")
print("---------------------")
astar_sol2 = astar(start_state2)
display_solution(astar_sol2)


---------------------
A* on Start Goal 1
---------------------
Expanding node 0 with g: 0, h: 16, f: 16
Expanding node 1 with g: 1, h: 17, f: 18
Expanding node 2 with g: 1, h: 17, f: 18
Expanding node 3 with g: 2, h: 16, f: 18
Expanding node 4 with g: 2, h: 16, f: 18
Expanding node 5 with g: 2, h: 16, f: 18
Expanding node 6 with g: 2, h: 16, f: 18
Expanding node 7 with g: 2, h: 16, f: 18
Expanding node 8 with g: 3, h: 15, f: 18
Expanding node 9 with g: 3, h: 15, f: 18
Expanding node 10 with g: 3, h: 15, f: 18
Expanding node 11 with g: 3, h: 15, f: 18
Expanding node 12 with g: 4, h: 14, f: 18
Expanding node 13 with g: 4, h: 14, f: 18
Expanding node 14 with g: 4, h: 14, f: 18
Expanding node 15 with g: 4, h: 14, f: 18
Expanding node 16 with g: 5, h: 13, f: 18
Expanding node 17 with g: 5, h: 13, f: 18
Expanding node 18 with g: 5, h: 13, f: 18
Expanding node 19 with g: 6, h: 12, f: 18
Expanding node 20 with g: 6, h: 12, f: 18
Expanding node 21 with g: 6, h: 12, f: 18
Expanding node 22 with

TypeError: 'int' object is not iterable